# Gate Semana 1 — WVC 2026 (§11)

**Objetivo:** medir o **gap bruto** de acurácia balanceada entre **Clássico-A (LBP multi-escala)** e **FM-ref congelado (Phikon / UNI)**, sob probe linear, em 100% / 5% (e 1%) de rótulo, e decidir **GO / NO-GO(→§10)**. Reproduzir um número de sanidade publicado para validar o pipeline.

## Decisões metodológicas deste notebook (registrar no log de pré-registro)

1. **O gate roda em split INTERNO do NCT-100K, não no 7k.** O §3 diz que o CRC-VAL-HE-7K é tocado *uma única vez, na tabela final*. Medir o gap no 7k na Semana 1 queimaria o avaliador honesto. Então: split estratificado do 100k (treino-gate / eval-gate, `INTERNAL_SPLIT_SEED` fixo). O 7k é **extraído e cacheado** (você vai precisar das features depois de qualquer forma), mas **não é pontuado** no gate.
   - **Caveat que não dá pra esconder:** o gap interno é *same-cohort*. Sob shift de coorte, o FM pode segurar e o LBP degradar → **o gap externo pode ser MAIOR que o interno**. Logo: gap interno grande ⇒ GO confiável; gap interno minúsculo ⇒ *sugestivo* de §10, **não conclusivo** (o 7k poderia reabrir o gap). O gate interno **não** é conservador — é rápido.

2. **FM-ref default = Phikon** (`owkin/phikon`), não UNI. UNI é *gated* no HuggingFace (precisa de aprovação do MahmoodLab); travar no `403` na Semana 1 é risco de execução gratuito. Troque para UNI via `ARM_FM = "uni"` se já tem acesso.

3. **A decisão-chave é o gap em 5%**, não em 100%. Gap pequeno em 100% é quase garantido pela saturação (~95%+) e **não** é gatilho de no-go sozinho — o §10 exige gap pequeno em 100% **E** em 5%. Adicionei 1% (custo desprezível) porque é o regime pro qual o §10 pivota. A decisão usa o **IC do gap**, não o ponto estimado.

4. **Número de sanidade: slot em branco.** Você registrou que ia sourcear a figura publicada independentemente (pra não citar número memorizado). A célula de sanidade **produz** o número; você preenche o alvo publicado e a fonte.


## 0. Dependências (rode uma vez)

In [1]:
!pip install torch torchvision scikit-image scikit-learn scipy pillow joblib tqdm transformers timm
# UNI (opcional) precisa de: huggingface_hub login + acesso aprovado a MahmoodLab/UNI.


## 1. Config `[edite os paths]`

In [2]:
from pathlib import Path
import json, hashlib, numpy as np

# --- Paths: cada diretório deve conter 9 subpastas (uma por classe) ---
NCT_DIR = Path("./NCT-CRC-HE-100K")   # pool de treino/probe (100k)
VAL_DIR = Path("./CRC-VAL-HE-7K")     # teste externo (7k) — extraído, NÃO pontuado no gate
CACHE_DIR = Path("./gate_cache"); CACHE_DIR.mkdir(exist_ok=True, parents=True)

# --- FM-ref ---
ARM_FM = "phikon"          # "phikon" (default, aberto) | "uni" (gated; precisa de acesso aprovado)
PHIKON_ID = "owkin/phikon" # v1, ViT-B, CLS token, 768-d. ("owkin/phikon-v2" -> ViT-L 1024-d)

# --- Regime de rótulo (§11 pede 100% e 5%; 1% incluído p/ contexto do §10) ---
LABEL_FRACTIONS = [1.0, 0.05, 0.01]
DECISIVE_FRAC   = 0.05     # frac que decide o go/no-go

# --- Split interno do 100k para o gate (registrado) ---
INTERNAL_SPLIT_SEED = 20260816   # data-alvo de submissão como seed mnemônico; NÃO mudar depois de olhar
INTERNAL_EVAL_FRAC  = 0.20       # fração do 100k reservada como eval-gate

# --- Probe ---
CLASS_WEIGHT = "balanced"  # aplicado IDÊNTICO aos dois braços; troque só se mudar nos dois
LOGREG_MAX_ITER = 2000

# --- Estatística do gate ---
N_SEEDS_GATE = 5           # variância aqui = variância da subamostragem de rótulo (extrator é fixo)
GAP_THRESHOLD = 2.0        # pontos de acurácia balanceada (escala 0-100), limiar do §10

# --- Extração ---
BATCH_SIZE = 128
NUM_WORKERS = 8
MAX_PER_CLASS = None       # ex.: 200 p/ um dry-run rápido de pipeline; None = tudo

# --- LBP (Clássico-A): multi-escala, uniforme, rotação-invariante, grayscale ---
LBP_PARAMS = [(8, 1), (16, 2), (24, 3)]   # (P, R) -> P+2 bins cada; dim total = sum(P+2) = 54

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| FM:", ARM_FM)


device: cuda | FM: phikon


## 2. Indexação dos datasets (9 pastas → listas (path, label))

In [3]:
IMG_EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"}

def index_folder(root: Path, max_per_class=None):
    root = Path(root)
    assert root.exists(), f"não existe: {root}"
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    if len(classes) != 9:
        print(f"[AVISO] {root.name}: {len(classes)} classes (esperado 9): {classes}")
    cls2idx = {c: i for i, c in enumerate(classes)}
    paths, labels = [], []
    for c in classes:
        files = sorted([p for p in (root / c).iterdir() if p.suffix.lower() in IMG_EXTS])
        if max_per_class: files = files[:max_per_class]
        paths.extend(files); labels.extend([cls2idx[c]] * len(files))
    return paths, np.array(labels), classes

train_paths, train_labels, CLASSES = index_folder(NCT_DIR, MAX_PER_CLASS)
val_paths,   val_labels,   VAL_CLASSES = index_folder(VAL_DIR, MAX_PER_CLASS)
assert CLASSES == VAL_CLASSES, f"ordem de classes difere!\n100k={CLASSES}\n7k={VAL_CLASSES}"
print("classes (ordem fixa):", CLASSES)
print(f"100k: {len(train_paths)} imgs | 7k: {len(val_paths)} imgs")
# Nota: BACK (background) é trivialmente separável e infla a acurácia dos DOIS braços igualmente.
# Mantido p/ comparabilidade com números publicados de 9 classes.


classes (ordem fixa): ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
100k: 100000 imgs | 7k: 7180 imgs


## 3. Clássico-A — LBP multi-escala (54-d)

In [4]:
from skimage.feature import local_binary_pattern
from PIL import Image
from joblib import Parallel, delayed

def lbp_feature(path):
    g = np.asarray(Image.open(path).convert("L"))  # uint8 grayscale (evita warning de float do LBP)
    feats = []
    for P, R in LBP_PARAMS:
        codes = local_binary_pattern(g, P, R, method="uniform")  # rot-inv uniforme -> P+2 bins
        n_bins = P + 2
        hist, _ = np.histogram(codes.ravel(), bins=n_bins, range=(0, n_bins), density=True)
        feats.append(hist.astype(np.float32))
    return np.concatenate(feats)

# NOTA de honestidade p/ o revisor de textura: LBP aqui é em GRAYSCALE — descarta a cor H&E,
# que é genuinamente discriminativa em histopatologia. Isso é a linha-base padrão e defensável
# do gate. Se o Clássico-A sair fraco, PARTE disso é cegueira de cor — que é assunto de
# Clássico-B / enriquecimento H3-b (§5), NÃO motivo pra "buffar" secretamente o baseline aqui.

def extract_lbp(paths, split_name):
    cache = CACHE_DIR / f"lbp_{split_name}_n{len(paths)}.npy"  # n<rows> no nome: cache velho de outro MAX_PER_CLASS nunca é reusado por engano
    if cache.exists():
        F = np.load(cache); print(f"[cache] LBP {split_name}: {F.shape}"); return F
    print(f"extraindo LBP {split_name} ({len(paths)} imgs)...")
    F = np.stack(Parallel(n_jobs=-1, verbose=5)(delayed(lbp_feature)(p) for p in paths))
    np.save(cache, F.astype(np.float32)); print(f"salvo {cache}: {F.shape}")
    return F


## 4. FM-ref — extrator congelado (Phikon default / UNI opcional)

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader

class PILListDataset(Dataset):
    def __init__(self, paths, preprocess):
        self.paths = paths; self.preprocess = preprocess
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.preprocess(Image.open(self.paths[i]).convert("RGB"))

def _load_hf(fn, *args, **kwargs):
    """from_pretrained com auto-recuperacao de checkpoint truncado no cache HF.
    Uma tentativa normal; se falhar (ex.: SafetensorError 'file not fully covered'),
    refaz UMA vez com force_download=True e re-levanta se ainda falhar."""
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print(f"[FM] carga falhou ({type(e).__name__}: {e}).")
        print(f"     Cache HF provavelmente truncado/corrompido -> refazendo com force_download=True...")
        return fn(*args, force_download=True, **kwargs)

def build_fm():
    "Retorna (forward_fn, preprocess_fn, feat_dim); forward_fn: batch_tensor -> [B, D] numpy."
    if ARM_FM == "phikon":
        from transformers import AutoImageProcessor, AutoModel
        proc = _load_hf(AutoImageProcessor.from_pretrained, PHIKON_ID)
        model = _load_hf(AutoModel.from_pretrained, PHIKON_ID).eval().to(DEVICE)
        def preprocess(img):
            return proc(img, return_tensors="pt")["pixel_values"][0]
        @torch.no_grad()
        def forward(batch):
            out = model(pixel_values=batch.to(DEVICE))
            cls = out.last_hidden_state[:, 0]          # CLS token
            return cls.float().cpu().numpy()
        dim = model.config.hidden_size
    elif ARM_FM == "uni":
        import timm
        from timm.data import resolve_data_config
        from timm.data.transforms_factory import create_transform
        # requer: huggingface_hub login + acesso aprovado a MahmoodLab/UNI
        # NOTA: sem auto-recuperacao aqui de proposito -- timm.create_model NAO aceita
        # force_download; se o checkpoint do UNI truncar, apague o cache manualmente:
        #   rm -rf ~/.cache/huggingface/hub/models--MahmoodLab--UNI
        model = timm.create_model("hf-hub:MahmoodLab/UNI", pretrained=True,
                                  init_values=1e-5, dynamic_img_size=True).eval().to(DEVICE)
        cfg = resolve_data_config({}, model=model)
        tfm = create_transform(**cfg)
        def preprocess(img): return tfm(img)
        @torch.no_grad()
        def forward(batch):
            return model(batch.to(DEVICE)).float().cpu().numpy()  # timm ViT -> features pooled (1024)
        dim = model.num_features
    else:
        raise ValueError(ARM_FM)
    return forward, preprocess, dim

def extract_fm(paths, split_name):
    cache = CACHE_DIR / f"fm_{ARM_FM}_{split_name}_n{len(paths)}.npy"  # idem: chave inclui numero de linhas
    if cache.exists():
        F = np.load(cache); print(f"[cache] FM({ARM_FM}) {split_name}: {F.shape}"); return F
    forward, preprocess, dim = build_fm()
    ds = PILListDataset(paths, preprocess)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)
    feats, n = [], len(paths)
    for bi, batch in enumerate(dl):
        feats.append(forward(batch))
        if bi % 20 == 0: print(f"  FM {split_name}: {min((bi+1)*BATCH_SIZE, n)}/{n}", end="\r")
    F = np.concatenate(feats, 0)
    np.save(cache, F.astype(np.float32)); print(f"\nsalvo {cache}: {F.shape}")
    return F


## 5. Rodar extração (cacheada). 7k é extraído mas NÃO pontuado no gate.

In [6]:
# 100k — usado no gate (split interno)
lbp_train = extract_lbp(train_paths, "train100k")
fm_train  = extract_fm(train_paths, "train100k")

# 7k — extraído e cacheado agora (reuso posterior); pontuado só na sanidade e na tabela final
lbp_val = extract_lbp(val_paths, "val7k")
fm_val  = extract_fm(val_paths, "val7k")

FEATS = {"LBP": (lbp_train, lbp_val), "FM": (fm_train, fm_val)}
for k,(a,b) in FEATS.items(): print(f"{k}: train {a.shape}  val {b.shape}")


extraindo LBP train100k (100000 imgs)...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.5s


[Parallel(n_jobs=-1)]: Done  72 tasks      | elapsed:    1.0s


[Parallel(n_jobs=-1)]: Done 432 tasks      | elapsed:    3.2s


...

[Parallel(n_jobs=-1)]: Done 6984 tasks      | elapsed:   43.7s


[Parallel(n_jobs=-1)]: Done 7180 out of 7180 | elapsed:   44.9s finished


salvo gate_cache/lbp_val7k_n7180.npy: (7180, 54)


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.



salvo gate_cache/fm_phikon_val7k_n7180.npy: (7180, 768)
LBP: train (100000, 54)  val (7180, 54)
FM: train (100000, 768)  val (7180, 768)


## 6. Probe linear + subamostragem estratificada por seed

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score

def train_eval_probe(Xtr, ytr, Xte, yte, seed=0):
    sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=LOGREG_MAX_ITER, class_weight=CLASS_WEIGHT, random_state=seed)
    clf.fit(sc.transform(Xtr), ytr)
    pred = clf.predict(sc.transform(Xte))
    return balanced_accuracy_score(yte, pred), f1_score(yte, pred, average="macro")

def stratified_subsample(y, frac, seed):
    "Indices de uma subamostra estratificada e reprodutivel."
    rng = np.random.RandomState(seed); y = np.asarray(y); idx = []
    for c in np.unique(y):
        ci = np.where(y == c)[0]
        k = max(1, int(round(len(ci) * frac)))
        idx.extend(rng.choice(ci, size=k, replace=False))
    return np.array(sorted(idx))


## 7. Split interno do 100k (registrado) — onde o gate mede o gap

In [8]:
rng = np.random.RandomState(INTERNAL_SPLIT_SEED)
# split estratificado treino-gate / eval-gate
gate_tr, gate_te = [], []
for c in np.unique(train_labels):
    ci = np.where(train_labels == c)[0]; rng.shuffle(ci)
    ne = int(round(len(ci) * INTERNAL_EVAL_FRAC))
    gate_te.extend(ci[:ne]); gate_tr.extend(ci[ne:])
gate_tr = np.array(sorted(gate_tr)); gate_te = np.array(sorted(gate_te))
print(f"treino-gate: {len(gate_tr)} | eval-gate: {len(gate_te)} (seed {INTERNAL_SPLIT_SEED})")


treino-gate: 80002 | eval-gate: 19998 (seed 20260816)


## 8. Experimento do gate — gap pareado por (frac, seed)

In [9]:
from scipy import stats

def ci95(vals):
    vals = np.asarray(vals, float); n = len(vals)
    m = vals.mean(); sem = vals.std(ddof=1)/np.sqrt(n) if n > 1 else 0.0
    t = stats.t.ppf(0.975, n-1) if n > 1 else 0.0
    return m, m - t*sem, m + t*sem

results = {}   # (arm, frac) -> list de bal_acc (%) por seed
gaps    = {}   # frac -> list de gap(%) por seed (FM - LBP, PAREADO no mesmo subsample)

for frac in LABEL_FRACTIONS:
    gaps[frac] = []
    # frac>=1.0: subamostra estratificada vira no-op -> treino IDÊNTICO em toda seed
    # (LogReg lbfgs é determinístico). Roda 1x; o "IC" em 100% é ponto, não intervalo.
    frac_seeds = range(N_SEEDS_GATE) if frac < 1.0 else range(1)
    for seed in frac_seeds:
        # MESMA subamostra p/ os dois braços -> gap pareado
        sub = stratified_subsample(train_labels[gate_tr], frac, seed)
        tr_idx = gate_tr[sub]
        per_arm = {}
        for arm in ("LBP", "FM"):
            Ftr, _ = FEATS[arm]
            ba, _ = train_eval_probe(Ftr[tr_idx], train_labels[tr_idx],
                                     Ftr[gate_te], train_labels[gate_te], seed=seed)
            results.setdefault((arm, frac), []).append(100*ba)
            per_arm[arm] = 100*ba
        gaps[frac].append(per_arm["FM"] - per_arm["LBP"])
    n_tr = len(stratified_subsample(train_labels[gate_tr], frac, 0))
    tag = "  [IC degenerado: 1 seed efetiva em 100%]" if frac >= 1.0 else ""
    print(f"\nfrac={frac:>5} (n_treino~{n_tr}):{tag}")
    for arm in ("LBP", "FM"):
        m, lo, hi = ci95(results[(arm, frac)])
        print(f"  {arm:3} bal_acc = {m:5.2f}%  [{lo:5.2f}, {hi:5.2f}]")
    gm, glo, ghi = ci95(gaps[frac])
    print(f"  GAP(FM-LBP) = {gm:+5.2f} pts  [{glo:+5.2f}, {ghi:+5.2f}]")



frac=  1.0 (n_treino~80002):  [IC degenerado: 1 seed efetiva em 100%]
  LBP bal_acc = 82.68%  [82.68, 82.68]
  FM  bal_acc = 99.78%  [99.78, 99.78]
  GAP(FM-LBP) = +17.11 pts  [+17.11, +17.11]



frac= 0.05 (n_treino~3999):
  LBP bal_acc = 80.26%  [80.12, 80.40]
  FM  bal_acc = 99.32%  [99.23, 99.42]
  GAP(FM-LBP) = +19.07 pts  [+19.01, +19.13]



frac= 0.01 (n_treino~800):
  LBP bal_acc = 76.25%  [75.79, 76.70]
  FM  bal_acc = 98.64%  [98.30, 98.98]
  GAP(FM-LBP) = +22.39 pts  [+22.02, +22.77]


## 9. Sanidade — reprodução de número publicado `[toque único sancionado do 7k]`

Treina o probe FM no **100k inteiro** e avalia **uma vez** no **7k inteiro** (protocolo dos papers publicados). Sem tuning contra este número — é validação de pipeline, não resultado de braço.

**Preencha o alvo publicado que você sourcear** (`PUBLISHED_TARGET`, `PUBLISHED_SOURCE`).

In [10]:
PUBLISHED_TARGET = 0.935    # ex.: 0.95  (acurácia balanceada publicada, probe linear, 9 classes, 7k)
PUBLISHED_SOURCE = "TODO: sourcear e registrar (autor, ano, tabela)"

ba_s, f1_s = train_eval_probe(fm_train, train_labels, fm_val, val_labels, seed=0)
print(f"[SANIDADE] FM({ARM_FM}) probe linear, treino=100k, teste=7k(externo):")
print(f"  bal_acc = {ba_s*100:.2f}%   macro-F1 = {f1_s*100:.2f}%")
if PUBLISHED_TARGET is not None:
    d = (ba_s - PUBLISHED_TARGET)*100
    print(f"  alvo publicado = {PUBLISHED_TARGET*100:.2f}% ({PUBLISHED_SOURCE}) | delta = {d:+.2f} pts")
    print("  -> pipeline OK" if abs(d) <= 2 else "  -> DIVERGÊNCIA >2pts: investigar antes de seguir")
else:
    print(f"  [!] PUBLISHED_TARGET não preenchido. Fonte: {PUBLISHED_SOURCE}")


[SANIDADE] FM(phikon) probe linear, treino=100k, teste=7k(externo):
  bal_acc = 93.02%   macro-F1 = 92.10%
  alvo publicado = 93.50% (TODO: sourcear e registrar (autor, ano, tabela)) | delta = -0.48 pts
  -> pipeline OK


## 10. Decisão GO / NO-GO (§10 / §11)

In [11]:
def gap_stats(frac): return ci95(gaps[frac])

g5_m,  g5_lo,  g5_hi  = gap_stats(DECISIVE_FRAC)
g100_m,g100_lo,g100_hi= gap_stats(1.0)

print(f"gap @5%   = {g5_m:+.2f} pts  [{g5_lo:+.2f}, {g5_hi:+.2f}]  (DECISIVO)")
print(f"gap @100% = {g100_m:+.2f} pts  [{g100_lo:+.2f}, {g100_hi:+.2f}]  (saturado; pouco informativo)")

# "sem sinal" = IC INTEIRO dentro de ±thr (consistente com ~0), nos DOIS regimes.
# A versão antiga só olhava o topo do IC -> um gap NEGATIVO grande (LBP>FM) passava
# como "topo < 2" e era rotulado "sem sinal", que é o oposto da verdade.
def near_zero(lo, hi): return (lo > -GAP_THRESHOLD) and (hi < GAP_THRESHOLD)

no_signal_both = near_zero(g5_lo, g5_hi) and near_zero(g100_lo, g100_hi)
strong_signal  = (g5_lo >= GAP_THRESHOLD) or (g5_hi <= -GAP_THRESHOLD)  # IC inteiro fora de ±thr, QUALQUER direção

print("\n" + "="*60)
if no_signal_both:
    print(f"NO-GO -> aciona §10. IC do gap em 100% E em 5% inteiro dentro de ±{GAP_THRESHOLD:.0f} pts")
    print("(consistente com ~0): o dataset não tem sinal p/ a pergunta original.")
elif strong_signal:
    if g5_m >= 0:
        print(f"GO. IC do gap em 5% inteiro fora de +{GAP_THRESHOLD:.0f} pts (FM>LBP): há gap p/ dissecar.")
    else:
        print(f"GO (achado forte). IC do gap em 5% inteiro < -{GAP_THRESHOLD:.0f} pts: LBP BATE o FM.")
        print("     Gap negativo NÃO é no-go — é manchete. Reporte o sinal (ver caveat c).")
else:
    print("AMBÍGUO. O IC do gap em 5% cruza o limiar. ANTES de decidir:")
    print(" - olhe o gap em 1% (regime pro qual o §10 pivota);")
    print(" - rode a decomposição por classe (H4): o sinal pode estar")
    print("   concentrado em classes morfológicas mesmo com gap agregado pequeno;")
    print(" - NÃO queime o 7k ainda.")
print("="*60)
print('''
CAVEATS (não ignorar):
 (a) Este gate é INTERNO (same-cohort). Sob shift de coorte o gap externo
     pode ser MAIOR -> gap interno minúsculo é sugestivo, não conclusivo.
 (b) 100% está saturado; o número que carrega a decisão é o de 5% (e 1%).
 (c) Gap NEGATIVO (LBP>FM) seria achado forte por si só — reporte o sinal.
''')


gap @5%   = +19.07 pts  [+19.01, +19.13]  (DECISIVO)
gap @100% = +17.11 pts  [+17.11, +17.11]  (saturado; pouco informativo)

GO. IC do gap em 5% inteiro fora de +2 pts (FM>LBP): há gap p/ dissecar.

CAVEATS (não ignorar):
 (a) Este gate é INTERNO (same-cohort). Sob shift de coorte o gap externo
     pode ser MAIOR -> gap interno minúsculo é sugestivo, não conclusivo.
 (b) 100% está saturado; o número que carrega a decisão é o de 5% (e 1%).
 (c) Gap NEGATIVO (LBP>FM) seria achado forte por si só — reporte o sinal.



---
*Ao terminar: registrar no log de pré-registro (§11) qual ramo foi tomado (GO / §10-1 / §10-2 / §10-3), com data, os três números de gap (1/5/100%) e o número de sanidade + fonte.*